# 🏃 User Fitness Profile Clustering
## Machine Learning Project — Nihed
### Objective: Segment users into fitness profiles using K-Means clustering

**Algorithm:** K-Means  
**Features:** 8 meaningful fitness & health indicators  
**Output:** 4 user segments — Faible performance / Sportifs occasionnels / Sportifs réguliers / Athlètes intensifs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
print("✅ Imports successful")

## 1. Load Dataset

In [ ]:
df = pd.read_csv('df_clean.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

## 2. Feature Selection
### Why these 8 features?

We select only **meaningful fitness indicators** that actually describe a user's fitness profile.
No meal names, cooking methods, or exercise equipment — those don't define who a person is as an athlete.

| Feature | Why it matters |
|---------|---------------|
| Age | Fitness capacity changes with age |
| BMI | Body composition indicator |
| Fat_Percentage | Direct body composition measure |
| Session_Duration | Commitment level |
| Workout_Frequency | Training consistency |
| Calories_Burned | Output intensity |
| Experience_Level | Training background |
| Avg_BPM | Cardiovascular fitness |
| Water_Intake | Recovery & discipline indicator |

In [ ]:
features = [
    'Age',
    'BMI',
    'Fat_Percentage',
    'Session_Duration (hours)',
    'Workout_Frequency (days/week)',
    'Calories_Burned',
    'Experience_Level',
    'Avg_BPM',
    'Water_Intake (liters)'
]

df_cluster = df[features].copy()

print(f'Working dataset shape: {df_cluster.shape}')
print(f'Missing values: {df_cluster.isnull().sum().sum()}')
print()
print(df_cluster.describe().round(2))

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

colors = ['#6366F1', '#10B981', '#F59E0B', '#EF4444',
          '#8B5CF6', '#EC4899', '#14B8A6', '#F97316', '#3B82F6']

for ax, feat, color in zip(axes.flat, features, colors):
    ax.hist(df_cluster[feat], bins=40, color=color, alpha=0.8, edgecolor='white', linewidth=0.4)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### 3.1 Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_cluster.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Scaling

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_cluster)
scaled_df = pd.DataFrame(scaled_data, columns=features)

print("✅ Scaled — mean ≈ 0, std ≈ 1 for all features")
print()
print(scaled_df.describe().round(3))

## 5. Finding Optimal K — Elbow Method

The **inertia** measures the sum of squared distances from each point to its cluster center.
We look for the "elbow" — the point where adding more clusters gives diminishing returns.

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(scaled_df)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(scaled_df, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Finding Optimal K', fontsize=13, fontweight='bold')

# Elbow
axes[0].plot(K_range, inertia, 'o-', color='#6366F1', linewidth=2, markersize=8)
axes[0].fill_between(K_range, inertia, alpha=0.1, color='#6366F1')
axes[0].axvline(4, color='#EF4444', linestyle='--', linewidth=2, label='K=4 (chosen)')
axes[0].set_title('Elbow Method — Inertia', fontweight='bold')
axes[0].set_xlabel('Number of Clusters K')
axes[0].set_ylabel('Inertia')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, silhouette_scores, 'o-', color='#10B981', linewidth=2, markersize=8)
axes[1].fill_between(K_range, silhouette_scores, alpha=0.1, color='#10B981')
axes[1].axvline(4, color='#EF4444', linestyle='--', linewidth=2, label='K=4 (chosen)')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of Clusters K')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].legend()

plt.tight_layout()
plt.show()

best_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f'Best K by silhouette: {best_k}')
print(f'Chosen K: 4 (balances interpretability and performance)')

## 6. Train K-Means (K=4)

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled_df)

df_cluster['Cluster'] = clusters

print(f'✅ K-Means trained with K={K}')
print()
print('Cluster sizes:')
sizes = df_cluster['Cluster'].value_counts().sort_index()
for c, n in sizes.items():
    print(f'  Cluster {c}: {n} users ({n/len(df_cluster)*100:.1f}%)')

## 7. Cluster Visualization with PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(scaled_df)

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['Cluster'] = clusters

explained = pca.explained_variance_ratio_
print(f'Variance explained: PC1={explained[0]*100:.1f}% | PC2={explained[1]*100:.1f}% | Total={sum(explained)*100:.1f}%')

palette = ['#EF4444', '#F59E0B', '#6366F1', '#10B981']
labels_map = {0: 'Faible performance', 1: 'Sportifs occasionnels',
              2: 'Sportifs réguliers',  3: 'Athlètes intensifs'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('K-Means Clustering — PCA Visualization', fontsize=13, fontweight='bold')

# Scatter by cluster
for c in range(K):
    mask = df_pca['Cluster'] == c
    axes[0].scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                    c=palette[c], label=labels_map[c], alpha=0.5, s=15)

# Plot centroids in PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='white', s=200, marker='*', zorder=5, edgecolors='black', label='Centroids')
axes[0].set_title(f'K-Means K={K} — PCA Projection', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[0].legend(fontsize=9)

# Cluster size bar chart
sizes = df_cluster['Cluster'].value_counts().sort_index()
bars = axes[1].bar([labels_map[i] for i in sizes.index], sizes.values,
                   color=palette, edgecolor='white', linewidth=1.2)
axes[1].set_title('Users per Cluster', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, sizes.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20, str(val),
                 ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Cluster Profiles Analysis

In [ ]:
cluster_means = df_cluster.groupby('Cluster')[features].mean().round(2)
cluster_means.index = [labels_map[i] for i in cluster_means.index]

print("=== Average values per cluster ===")
print(cluster_means.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(12, 5))
cluster_norm = cluster_means.copy()
for col in cluster_norm.columns:
    cluster_norm[col] = (cluster_norm[col] - cluster_norm[col].min()) /                         (cluster_norm[col].max() - cluster_norm[col].min())

sns.heatmap(cluster_norm, annot=cluster_means.values, fmt='.1f',
            cmap='RdYlGn', linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Cluster Profiles — Normalized Heatmap (values = real averages)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

### 8.1 Radar Chart — Cluster Comparison

In [ ]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Normalize for radar
radar_df = cluster_means.copy()
for col in radar_df.columns:
    radar_df[col] = (radar_df[col] - radar_df[col].min()) /                     (radar_df[col].max() - radar_df[col].min() + 1e-9)

categories = features
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_facecolor('#080C14')
fig.patch.set_facecolor('#080C14')

for i, (idx, row) in enumerate(radar_df.iterrows()):
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, color=palette[i], label=idx)
    ax.fill(angles, values, alpha=0.15, color=palette[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=9, color='white')
ax.set_yticklabels([])
ax.set_title('Cluster Profiles — Radar Chart', size=13,
             fontweight='bold', color='white', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.spines['polar'].set_color('#1A2035')
ax.grid(color='#1A2035', linewidth=0.8)

plt.tight_layout()
plt.show()

## 9. Cluster Interpretation

| Cluster | Profile | Description |
|---------|---------|-------------|
| 0 | 🔴 Faible performance | Low calories burned, short sessions, low frequency. Beginners or sedentary users. |
| 1 | 🟡 Sportifs occasionnels | Moderate activity, average BMI. Exercise occasionally but inconsistently. |
| 2 | 🔵 Sportifs réguliers | Good frequency, moderate intensity. Consistent but not elite performers. |
| 3 | 🟢 Athlètes intensifs | High calories burned, long sessions, high frequency, low BMI. Elite performers. |

These segments can be used to:
- Personalize workout recommendations
- Target nutritional advice per profile
- Track user progression between clusters

## 10. Predict Cluster for New User

In [ ]:
def predict_cluster(age, bmi, fat_pct, duration, frequency,
                    calories, experience, avg_bpm, water):

    labels_map_local = {
        0: '🔴 Faible performance',
        1: '🟡 Sportifs occasionnels',
        2: '🔵 Sportifs réguliers',
        3: '🟢 Athlètes intensifs'
    }

    input_data = np.array([[age, bmi, fat_pct, duration,
                             frequency, calories, experience,
                             avg_bpm, water]])

    input_scaled = scaler.transform(input_data)
    cluster = kmeans.predict(input_scaled)[0]

    print(f'Predicted Cluster: {cluster} — {labels_map_local[cluster]}')
    return cluster

# Example
predict_cluster(
    age=28, bmi=22.5, fat_pct=15.0,
    duration=1.5, frequency=5,
    calories=700, experience=2,
    avg_bpm=150, water=3.0
)

## 11. Export Models

In [ ]:
import joblib, os

os.makedirs('models/nihed', exist_ok=True)

joblib.dump(kmeans, 'models/nihed/kmeans.pkl')
joblib.dump(scaler, 'models/nihed/scaler.pkl')

print('✅ Saved: models/nihed/kmeans.pkl')
print('✅ Saved: models/nihed/scaler.pkl')
print()
print(f'KMeans: {K} clusters, trained on {len(df_cluster)} users')
print(f'Features: {features}')